In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# LDBC SNB RESULTS ANALYSIS — CORRECTED VERSION
# =========================================================
# Corrections aligned with the IMDb corrected notebook:
#
# 1) Metrics are computed per:
#    query_name + scale_label + run_phase
#
# 2) Main comparison metric:
#    p95_latency_ms
#
# 3) Top-1 preservation:
#    checks whether the best configuration class in the full benchmarked
#    space belongs to the activated family.
#
# 4) Near-best preservation:
#    checks whether at least one activated configuration is within 5%
#    of the best full-space p95 latency.
#
# 5) Output artifacts:
#    ldbc_snb_summary_hot_by_scale.csv
#    ldbc_snb_diff_best_vs_primary_hot_by_scale.csv
#    ldbc_snb_best_by_query_scale_hot.csv
# =========================================================

# ---------------------------------------------------------
# 1) Configuration
# ---------------------------------------------------------

results_dir = Path(
    "/path/to/schemalens/ldbc_snb/results/ldbc_snb_sf3_full_fiben_format_clean"
)

scale_label_to_force = "sf3"   # change to sf0.1, sf1, or sf3
run_phase_to_analyze = "hot"

results_csv = results_dir / "benchmark_aggregate_results.csv"

output_dir = results_dir / "ldbc_snb_analysis_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# LDBC broader template space:
# G0, G1, G2, G3, G4, G5, G6, G7, G8, G9 = 10
n_C = 10

# Near-best threshold used in the paper/methodology
near_best_threshold = 0.05

# ---------------------------------------------------------
# 2) Read benchmark aggregate results
# ---------------------------------------------------------

agg = pd.read_csv(results_csv)

# ---------------------------------------------------------
# 3) Normalize column names from our FIBEN-style runner
# ---------------------------------------------------------

if "final_benchmark_group" not in agg.columns and "benchmark_group" in agg.columns:
    agg["final_benchmark_group"] = agg["benchmark_group"]

if "design_pattern" not in agg.columns:
    if "mongodb_pattern" in agg.columns:
        agg["design_pattern"] = agg["mongodb_pattern"]
    elif "document_strategy" in agg.columns:
        agg["design_pattern"] = agg["document_strategy"]
    else:
        agg["design_pattern"] = "unknown"

if "g_class" not in agg.columns:
    raise ValueError("Missing required column: g_class")

if "scale_label" not in agg.columns:
    agg["scale_label"] = scale_label_to_force
else:
    # Force correct label when analyzing one result folder
    agg["scale_label"] = scale_label_to_force

if "official_id" not in agg.columns:
    agg["official_id"] = agg["query_name"].str.extract(r"^(INS\d+|IC\d+|IS\d+)")

def infer_query_group_from_official_id(official_id):
    official_id = str(official_id)
    if official_id.startswith("IC"):
        return "complex_read"
    if official_id.startswith("IS"):
        return "short_read"
    if official_id.startswith("INS"):
        return "insert"
    return "other"

if "query_group" not in agg.columns:
    agg["query_group"] = agg["official_id"].apply(infer_query_group_from_official_id)

required_cols = [
    "candidate_id",
    "g_class",
    "final_benchmark_group",
    "design_pattern",
    "scale_label",
    "query_name",
    "query_group",
    "run_phase",
    "p95_latency_ms",
]

missing = [c for c in required_cols if c not in agg.columns]
if missing:
    raise ValueError(f"Missing required columns in benchmark_aggregate_results.csv: {missing}")

# ---------------------------------------------------------
# 4) Restrict to selected run phase
# ---------------------------------------------------------

phase_df = agg[agg["run_phase"] == run_phase_to_analyze].copy()

if phase_df.empty:
    raise ValueError(f"No rows found for run_phase = {run_phase_to_analyze!r}")

# ---------------------------------------------------------
# 5) Query-level corrected analysis
# ---------------------------------------------------------

rows = []

group_cols = ["query_name", "scale_label", "run_phase"]

for (query_name, scale_label, run_phase), grp in phase_df.groupby(group_cols):
    grp = grp.copy()

    # Best configuration in the full benchmarked comparison space
    best_all = grp.loc[grp["p95_latency_ms"].idxmin()]
    best_latency_full = float(best_all["p95_latency_ms"])
    best_config_full = str(best_all["g_class"])

    # Activated family = all non-control rows
    activated = grp[grp["final_benchmark_group"] != "control"].copy()
    activated_classes = set(
        activated["g_class"].dropna().astype(str).unique()
    )

    n_tested_configs = grp["g_class"].nunique()
    n_activated_configs = activated["g_class"].nunique()

    dsr = 1 - (n_activated_configs / n_C)

    # Top-1 preservation:
    # best full-space class belongs to activated family
    top1_preserved_by_activated = best_config_full in activated_classes

    # Near-best preservation:
    # at least one activated config is within 5% of the best p95
    near_best_mask = (
        (grp["p95_latency_ms"] - best_latency_full) / best_latency_full
    ) <= near_best_threshold

    near_best_classes = set(
        grp.loc[near_best_mask, "g_class"]
        .dropna()
        .astype(str)
        .unique()
    )

    near_best_preserved_by_activated = (
        len(activated_classes.intersection(near_best_classes)) > 0
    )

    # Activated regret
    if activated.empty:
        activated_regret = np.nan
        best_activated_config = None
        best_activated_p95_ms = np.nan
    else:
        best_activated = activated.loc[activated["p95_latency_ms"].idxmin()]
        best_activated_config = best_activated["g_class"]
        best_activated_p95_ms = float(best_activated["p95_latency_ms"])
        activated_regret = (
            best_activated_p95_ms - best_latency_full
        ) / best_latency_full

    # Primary-only regret
    primary = grp[grp["final_benchmark_group"] == "primary"].copy()

    if primary.empty:
        best_primary_config = None
        best_primary_p95_ms = np.nan
        primary_regret = np.nan
    else:
        best_primary = primary.loc[primary["p95_latency_ms"].idxmin()]
        best_primary_config = best_primary["g_class"]
        best_primary_p95_ms = float(best_primary["p95_latency_ms"])
        primary_regret = (
            best_primary_p95_ms - best_latency_full
        ) / best_latency_full

    rows.append({
        "official_id": best_all["official_id"],
        "query_name": query_name,
        "scale_label": scale_label,
        "run_phase": run_phase,
        "query_group": best_all["query_group"],

        "n_tested_configs": int(n_tested_configs),
        "n_activated_configs": int(n_activated_configs),
        "DSR": float(dsr),

        "best_config": best_all["g_class"],
        "best_group": best_all["final_benchmark_group"],
        "best_design_pattern": best_all["design_pattern"],
        "best_p95_ms": best_latency_full,

        "top1_preserved_by_activated": bool(top1_preserved_by_activated),
        "near_best_preserved_by_activated": bool(near_best_preserved_by_activated),
        "activated_regret": float(activated_regret) if pd.notna(activated_regret) else np.nan,

        "best_activated_config": best_activated_config,
        "best_activated_p95_ms": best_activated_p95_ms,

        "best_primary_config": best_primary_config,
        "best_primary_p95_ms": best_primary_p95_ms,
        "primary_regret": float(primary_regret) if pd.notna(primary_regret) else np.nan,
    })

analysis_df = (
    pd.DataFrame(rows)
    .sort_values(by=["official_id", "scale_label", "run_phase"])
    .reset_index(drop=True)
)

print(f"LDBC SNB query-level summary using run_phase = {run_phase_to_analyze!r}\n")
display(analysis_df)

# ---------------------------------------------------------
# 6) Global metrics
# ---------------------------------------------------------

print("Average DSR:", analysis_df["DSR"].mean())
print("Top-1 preservation activated:", analysis_df["top1_preserved_by_activated"].mean())
print("Near-best preservation activated:", analysis_df["near_best_preserved_by_activated"].mean())
print("Mean activated regret:", analysis_df["activated_regret"].dropna().mean())
print("Mean primary regret:", analysis_df["primary_regret"].dropna().mean())

print("\nBest group counts:")
print(analysis_df["best_group"].value_counts())

# ---------------------------------------------------------
# 7) Secondary-affected winners
# ---------------------------------------------------------

secondary_winners_df = analysis_df[
    analysis_df["best_group"] == "secondary_affected"
][
    [
        "official_id",
        "query_name",
        "scale_label",
        "run_phase",
        "best_config",
        "best_design_pattern",
        "best_p95_ms",
        "best_primary_config",
        "best_primary_p95_ms",
        "primary_regret",
    ]
].reset_index(drop=True)

print("\nSecondary-affected winners:")
display(secondary_winners_df)

# ---------------------------------------------------------
# 8) Control winners
# ---------------------------------------------------------

control_winners_df = analysis_df[
    analysis_df["best_group"] == "control"
][
    [
        "official_id",
        "query_name",
        "scale_label",
        "run_phase",
        "best_config",
        "best_design_pattern",
        "best_p95_ms",
        "best_primary_config",
        "best_primary_p95_ms",
        "primary_regret",
        "activated_regret",
    ]
].reset_index(drop=True)

print("\nControl winners:")
display(control_winners_df)

# ---------------------------------------------------------
# 9) Best configuration by query and scale
# ---------------------------------------------------------

best_by_query_scale_rows = []

for (query_name, scale_label), grp in phase_df.groupby(["query_name", "scale_label"]):
    best = grp.loc[grp["p95_latency_ms"].idxmin()]
    best_by_query_scale_rows.append({
        "official_id": best["official_id"],
        "query_name": query_name,
        "scale_label": scale_label,
        "run_phase": run_phase_to_analyze,
        "best_config": best["g_class"],
        "best_group": best["final_benchmark_group"],
        "best_design_pattern": best["design_pattern"],
        "best_p95_ms": best["p95_latency_ms"],
        "best_avg_latency_ms": best.get("avg_latency_ms", np.nan),
        "best_p99_ms": best.get("p99_latency_ms", np.nan),
        "avg_documents_returned": best.get("avg_documents_returned", np.nan),
        "avg_documents_written": best.get("avg_documents_written", np.nan),
    })

best_by_query_scale_df = (
    pd.DataFrame(best_by_query_scale_rows)
    .sort_values(by=["official_id", "scale_label"])
    .reset_index(drop=True)
)

display(best_by_query_scale_df)

# ---------------------------------------------------------
# 10) Export artifacts
# ---------------------------------------------------------

analysis_df.to_csv(
    output_dir / f"ldbc_snb_summary_{run_phase_to_analyze}_by_scale.csv",
    index=False
)

secondary_winners_df.to_csv(
    output_dir / f"ldbc_snb_diff_best_vs_primary_{run_phase_to_analyze}_by_scale.csv",
    index=False
)

best_by_query_scale_df.to_csv(
    output_dir / f"ldbc_snb_best_by_query_scale_{run_phase_to_analyze}.csv",
    index=False
)

control_winners_df.to_csv(
    output_dir / f"ldbc_snb_control_winners_{run_phase_to_analyze}_by_scale.csv",
    index=False
)

# Compatibility copy with previous file name
analysis_df.to_csv(
    results_dir / "schemalens_reduction_analysis_hot.csv",
    index=False
)

print(f"\nSaved outputs in: {output_dir.resolve()}")
print(f"Compatibility file saved in: {results_dir / 'schemalens_reduction_analysis_hot.csv'}")